# Lung Cancer Post-Translational Modification and Gene Expression Regulation

This notebook reproduces the [CST_Data_Viz.ipynb](https://nbviewer.org/github/MaayanLab/CST_Lung_Cancer_Viz/blob/master/notebooks/CST_Data_Viz.ipynb) case study from the [MaayanLab/CST_Lung_Cancer_Viz](https://github.com/MaayanLab/CST_Lung_Cancer_Viz) repository, originally visualized with [Clustergrammer](https://doi.org/10.1038/sdata.2017.151), using Celldega. Most of the text is carried over from the original notebook. We build each Clustergram by stacking modalities of a MuData object, and add two new views: a Clustergram for each PTM type, and a comparison of the groups of cell lines that each omic finds.

Lung cancer is a complex disease that is known to be regulated at the post-translational modification level, e.g. phosphorylation driven by kinases. Our collaborators at [Cell Signaling Technology Incorporated (CST)](https://www.cellsignal.com/) used Tandem Mass Tag (TMT) mass spectrometry to measure differential phosphorylation, acetylation, and methylation in a panel of 42 lung cancer cell lines compared to non-cancerous lung tissue. Gene expression data from 37 of these lung cancer cell lines was also independently obtained from the publicly available Cancer Cell Line Encyclopedia [(CCLE)](https://sites.broadinstitute.org/ccle/). In this notebook we will analyze PTM and gene expression regulation across these 37 common lung cancer cell lines. The post-translational modification (PTM) and gene expression data were pre-processed (normalized, filtered, etc.) in the original [CST_Data_Processing.ipynb](https://nbviewer.org/github/MaayanLab/CST_Lung_Cancer_Viz/blob/master/notebooks/CST_Data_Processing.ipynb) notebook.

In this notebook we will
* visualize PTM, Expression, and merged PTM-Expression datasets
* compare the clusters of cell lines found by each PTM type and by gene expression
* identify clusters of PTMs/genes
* use enrichment analysis to understand the biological processes involved in gene-expression clusters

### Lung Cancer Histology
The lung cancer cell lines in this joint dataset fall into two histologies: Non-small Cell Lung Cancer (NSCLC), and Small Cell Lung Cancer (SCLC). NSCLC is more common than SCLC and refers to all epithelial lung cancers other than SCLC. SCLC is highly malignant and develops metastasis earlier than NSCLC. SCLC is thought to originate from neuroendocrine cells and is known to have neuronal characteristics [(Onganer et al. 2005)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2361510/).

### Load Data
We use the processed PTM and gene expression data from the [Clustergrammer Lung Cancer PTM Case Study Data](https://doi.org/10.6084/m9.figshare.5339689) Figshare dataset (CC BY 4.0), adapted into a [MuData](https://mudata.scverse.org/) object, a more modern format for multimodal data, by the [CST_Lung_Cancer_PTM_GEX_Pre-process.ipynb](https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/blob/main/CST_Lung_Cancer_PTM_GEX_Pre-process.ipynb) notebook. Both are hosted with Celldega's [supporting data](https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data) on Hugging Face. The MuData has one modality per data type: phosphorylation (`phospho`), arginine methylation (`Rme1`), lysine acetylation (`AcK`), lysine methylation (`Kme1`), and gene expression (`rna`). Each modality has the 37 cell lines as observations, and the cell line annotations (histology, sub-histology, and TP53, EGFR, RB1, and KRAS mutation status) are in the shared `obs`.

In [1]:
from pathlib import Path
import shutil
from urllib.request import Request, urlopen

import anndata as ad
import celldega as dega
from ipywidgets import HBox, Label, Layout, VBox
import mudata as md
import pandas as pd

In [2]:
H5MU_URL = "https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/CST_Lung_Cancer_PTM_GEX.h5mu"

# Reuse the repository's notebooks/data folder when running from a checkout;
# otherwise cache the download beside the notebook.
working_dir = Path.cwd().resolve()
project_root = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "pyproject.toml").is_file() and (directory / "notebooks").is_dir()
    ),
    None,
)
data_dir = (project_root / "notebooks" if project_root else working_dir) / "data"
data_dir = data_dir / "Clustergrammer_Lung_Cancer_Case_Study_PTM_Data"
data_dir.mkdir(parents=True, exist_ok=True)

h5mu_path = data_dir / "CST_Lung_Cancer_PTM_GEX.h5mu"
if not h5mu_path.is_file():
    request = Request(H5MU_URL, headers={"User-Agent": "celldega-example-notebook"})
    with urlopen(request, timeout=300) as response, h5mu_path.open("wb") as output:
        shutil.copyfileobj(response, output)

# Adopt the upcoming MuData default: update() no longer pulls modality columns.
md.set_options(pull_on_update=False)
mdata = md.read_h5mu(h5mu_path)
cell_lines = mdata.obs
print(mdata)

MuData object with n_obs × n_vars = 37 × 2730
  obs:	'Histology', 'Sub-Histology', 'mut-TP53', 'mut-EGFR', 'mut-RB1', 'mut-KRAS'
  uns:	'source'
  5 modalities
    phospho:	37 x 1085
      var:	'Data-Type', 'gene'
    Rme1:	37 x 368
      var:	'Data-Type', 'gene'
    AcK:	37 x 226
      var:	'Data-Type', 'gene'
    Kme1:	37 x 51
      var:	'Data-Type', 'gene'
    rna:	37 x 1000
      var:	'Data-Type', 'gene'


The cell lines fall into two histologies and, within NSCLC, several sub-histologies. NSCLC cell lines without a more specific annotation are labeled `NSCLC`.

In [3]:
pd.crosstab(cell_lines["Sub-Histology"], cell_lines["Histology"])

Histology,NSCLC,SCLC
Sub-Histology,,
NSCLC,8,0
SCLC,0,8
adenocarcinoma,13,0
bronchioloalveolar_adenocarcinoma,4,0
large_cell_carcinoma,2,0
squamous_cell_carcinoma,2,0


### Stacking Modalities into Clustergrams
Since the datasets were processed similarly (e.g. Z-scored across cell lines), we can look for cross-data-type clusters by stacking modalities vertically. The `stack` function below concatenates modalities feature-wise with `anndata.concat`, so that PTMs and genes become the rows of a Clustergram and the cell lines become its columns. `clustergram_matrix` then builds and clusters a Celldega `Matrix`, using cosine distance and average linkage as in the original notebook. The values are already Z-scored, so they are clustered as is. We also reuse the original notebook's category colors: data types color the rows, while histology, sub-histology, and mutation status color the columns.

In [4]:
PTM_TYPES = ["phospho", "Rme1", "AcK", "Kme1"]
COL_ATTR = ["Histology", "Sub-Histology", "mut-TP53", "mut-EGFR", "mut-RB1", "mut-KRAS"]

# The original notebook's category colors, as hex codes.
CATEGORY_COLORS = {
    # Data types (rows)
    "phospho": "#FF0000",  # red
    "Rme1": "#800080",  # purple
    "AcK": "#0000FF",  # blue
    "Kme1": "#808080",  # grey
    "rna": "#FFFF00",  # yellow
    # Histology and sub-histology (columns)
    "SCLC": "#FF0000",
    "NSCLC": "#0000FF",
    "squamous_cell_carcinoma": "#FFFF00",
    "bronchioloalveolar_adenocarcinoma": "#FFA500",  # orange
    "adenocarcinoma": "#808080",
    "large_cell_carcinoma": "#008000",  # green; not set in the original
    # Mutation status (columns)
    "true": "#333333",
    "false": "#E0E0E0",
}


def stack(*modalities):
    # Concatenate modalities feature-wise: features become rows, cell lines columns.
    adata = ad.concat([mdata.mod[name] for name in modalities], axis="var")
    adata.obs = cell_lines.loc[adata.obs_names]
    return adata


def clustergram_matrix(adata, name):
    # Each Clustergram is keyed by its Matrix name, so every matrix gets a distinct one.
    mat = dega.clust.Matrix(
        adata,
        col_attr=COL_ATTR,
        row_attr=["Data-Type"],
        global_colors=CATEGORY_COLORS,
        name=name,
    )
    return mat.cluster()

## Clustergrams for Each PTM Type
Before stacking the PTM types, we cluster each one on its own. Phosphorylation was measured at the most sites, followed by arginine methylation, lysine acetylation, and lysine methylation.

In [5]:
ptm_type_mats = {ptm_type: clustergram_matrix(stack(ptm_type), ptm_type) for ptm_type in PTM_TYPES}
ptm_type_panels = [
    VBox([
        Label(f"{ptm_type}: {mat.data.shape[0]:,} PTMs"),
        dega.viz.Clustergram(matrix=mat, width=580, height=550, layout=Layout(width="690px")),
    ])
    for ptm_type, mat in ptm_type_mats.items()
]

In [6]:
VBox([HBox(ptm_type_panels[:2]), HBox(ptm_type_panels[2:])])

Phosphorylation separates SCLC (red) from NSCLC (blue) cell lines on its own. The methylation and acetylation types also bring most SCLC cell lines together (six of the eight SCLC cell lines sit side by side in the Rme1 and AcK Clustergrams), but the top split of their column dendrograms does not follow histology. We can check this by cutting each column dendrogram into its top two branches with `Matrix.cut_tree`, the programmatic counterpart of the Clustergram's dendrogram slider:

In [7]:
def column_branches(mat, n_clusters=2, by="Histology"):
    # Cell lines in the top branches of the column dendrogram, by a cell line annotation.
    branches = mat.cut_tree(axis="col", n_clusters=n_clusters)
    return pd.crosstab(branches, cell_lines.loc[branches.index, by])


pd.concat(
    {ptm_type: column_branches(mat) for ptm_type, mat in ptm_type_mats.items()},
    names=["Data-Type", "branch"],
)

Histology         NSCLC  SCLC
Data-Type branch             
phospho   1           1     6
          2          28     2
Rme1      1          11     2
          2          18     6
AcK       1           6     6
          2          23     2
Kme1      1          15     2
          2          14     6

Only phosphorylation puts six of the eight SCLC cell lines in a branch of their own, joined by the NSCLC cell line H2106.

## Comparing Cell Line Clusters Across Omics
Before stacking the omics, we can get a coarse-grained preview of whether they find the same groups of cell lines. We cut the column dendrogram of each modality (the four PTM types and gene expression) into the same number of clusters, and turn every cluster into a row that marks its cell lines with 1. Clustering these rows brings together clusters from different omics that contain similar sets of cell lines, while clustering the columns groups cell lines that are often clustered together across omics.

In [8]:
N_CLUSTERS = 5
mat_rna = clustergram_matrix(stack("rna"), "rna")
omic_mats = {**ptm_type_mats, "rna": mat_rna}


def cluster_membership(omic, mat, n_clusters):
    # One row per column-dendrogram cluster, marking its cell lines with 1.
    clusters = mat.cut_tree(axis="col", n_clusters=n_clusters)
    membership = pd.get_dummies(clusters, dtype=float)
    membership.columns = [f"{omic}-{label}" for label in membership.columns]
    return ad.AnnData(membership, var=pd.DataFrame({"Data-Type": omic}, index=membership.columns))


omic_clusters = ad.concat(
    [cluster_membership(omic, mat, N_CLUSTERS) for omic, mat in omic_mats.items()],
    axis="var",
)
omic_clusters.obs = cell_lines.loc[omic_clusters.obs_names]
mat_omic_clusters = clustergram_matrix(omic_clusters, "omic_clusters")
cgm_omic_clusters = dega.viz.Clustergram(matrix=mat_omic_clusters, width=700, height=600)

In [9]:
cgm_omic_clusters

Clustergram(category_colors={'phospho': '#FF0000', 'Rme1': '#800080', 'AcK': '#0000FF', 'Kme1': '#808080', 'rn…

The cluster numbers in the row labels (e.g. `phospho-1`) are arbitrary labels from each cut. The top of the row dendrogram collects one SCLC cluster from each omic: phosphorylation, lysine acetylation, lysine methylation, and gene expression each place the six RB1-mutant SCLC cell lines (DMS153, H209, H524, H526, H69, and H82) in one cluster together with H2106, while arginine methylation groups four of them with DMS53 and H446. The omics also agree on groups of NSCLC cell lines. Gene expression, phosphorylation, and arginine methylation each find a cluster of adenocarcinoma and bronchioloalveolar adenocarcinoma cell lines that includes Calu-3, H1650, H1781, H441, and HCC78. Arginine methylation and lysine acetylation find nearly the same cluster of CAL-12T, H1437, H1703, H2342, HCC44, and Lou-NH91, which phosphorylation partly recovers. Changing `N_CLUSTERS` shows how these shared groups hold up at coarser or finer cuts.

Next, we stack the omics, as the original notebook did, to look at the PTMs and genes behind these groups.

## CST Post Translational Modification Lung Cancer Data
Here we will visualize pre-processed PTM data from our collaborators at CST. In this dataset differential phosphorylation, methylation, and acetylation were measured relative to non-cancerous lung tissue across 37 lung cancer cell lines (note, not all PTM types had the same number of measurements, e.g. phosphorylation had the most measurements). PTM levels were quantile normalized in each cell line, PTMs with more than 7 missing values were discarded, and PTM levels across cell lines were Z-score normalized to highlight differential regulation across lung cancer cell lines. Cell line 'meta-data' is shown as column categories and include: Histology, Sub-Histology, TP53 mutation, EGFR mutation, RB1 mutation, and KRAS mutation. Cell line metadata was obtained from our collaborators at CST and from metadata available from the CCLE.

We stack the four PTM modalities into a single Clustergram of 1,730 PTMs. The `Data-Type` row category shows which modality each PTM comes from.

In [10]:
mat_ptm = clustergram_matrix(stack(*PTM_TYPES), "ptm")
cgm_ptm = dega.viz.Clustergram(matrix=mat_ptm, width=700, height=750)

In [11]:
cgm_ptm

Clustergram(category_colors={'phospho': '#FF0000', 'Rme1': '#800080', 'AcK': '#0000FF', 'Kme1': '#808080', 'rn…

### Cell Lines Cluster According to Histology and Mutation
Above, we see that cell lines cluster according to their histology - almost all NSCLC cell lines (blue column category) cluster together (except for H2106) and almost all SCLC cell lines (red column category) cluster together. The exceptions are DMS53 and H446, the two SCLC cell lines without an RB1 mutation, which sit next to the SCLC cluster but join the NSCLC branch at the top of the column dendrogram. Cell lines also cluster to some extent based on their Sub-Histology (e.g. squamous cell carcinoma cells in yellow and bronchioloalveolar adenocarcinoma in orange) and based on mutation. We also see that cell lines appear to cluster according to common mutations in: EGFR, KRAS, and RB1 (for example, the three EGFR-mutant cell lines H1650, H1975, and HCC827 are neighbors). Mutations in these genes may be the drivers behind their common PTM regulation.

### PTM Clusters
We can see two high-level clusters of PTMs that have either high/low levels in SCLC cell lines and low/high levels, respectively in NSCLC cell lines. The cluster with high levels in SCLC cell lines is mainly composed of phosphorylation, arginine methylation, and lysine acetylation, while the cluster with low levels in SCLC cell lines is almost entirely composed of phosphorylation (red row category). Clicking the row dendrogram selects these clusters in the Clustergram. Below, we cut the same dendrogram into its top two clusters with `Matrix.cut_tree`, count their data types, and compare their mean levels in NSCLC and SCLC cell lines.

In [12]:
def row_cluster_summary(mat, n_clusters=2):
    # Data types in the top row clusters, and each cluster's mean Z-score by histology.
    clusters = mat.cut_tree(axis="row", n_clusters=n_clusters)
    summary = pd.crosstab(clusters, mat.meta_row.loc[clusters.index, "Data-Type"])
    for histology, lines in cell_lines.groupby("Histology", observed=True).groups.items():
        summary[f"mean in {histology}"] = mat.data[lines].groupby(clusters).mean().mean(axis=1).round(2)
    return summary


row_cluster_summary(mat_ptm)

Data-Type,AcK,Kme1,Rme1,phospho,mean in NSCLC,mean in SCLC
cluster,,,,,,
1,18,2,59,613,0.13,-0.48
2,208,49,309,472,-0.09,0.31


### PTMs Cluster According to Data-Type
We see that PTMs (rows) also tend to cluster according to their type (e.g. phosphorylation). However, we also see that different types of modifications co-cluster. The cluster with low levels in SCLC cell lines is 89% phosphorylation, while the cluster with high levels in SCLC cell lines holds most of the arginine methylation, lysine acetylation, and lysine methylation sites.

## Gene Expression Data
We obtained gene expression data from the Cancer Cell Line Encyclopedia [(CCLE)](https://sites.broadinstitute.org/ccle/) for 37 lung cancer cell lines assayed by our collaborators at CST. This independent dataset can be used to find novel correlations between differentially expressed genes and PTMs as well as determine whether lung cancer cell lines behave similarly in gene-expression-space and PTM-space. The gene expression data was processed in the [CST_Data_Processing.ipynb](https://nbviewer.org/github/MaayanLab/CST_Lung_Cancer_Viz/blob/master/notebooks/CST_Data_Processing.ipynb) notebook that: kept the top 1000 genes with the greatest variance across the cell lines, and Z-score normalized the genes across the cell lines to highlight differential expression across the lung cancer cell lines.

We reuse the gene expression `Matrix` (`mat_rna`) from the cluster comparison above and link its Clustergram to an Enrich widget, which we describe below.

In [13]:
cgm_rna = dega.viz.Clustergram(matrix=mat_rna, width=490, height=700)
rna_enrich = dega.viz.clustergram_enrich(cgm_rna)
enrich = rna_enrich.children[1]
# Start Enrich on the Gene Ontology library used in the original analysis.
enrich.inst_lib = "GO_Biological_Process_2025"
# Match the Clustergram's full height, which includes its control panel.
enrich.height = 900

In [14]:
rna_enrich

### Cell Lines Cluster According to Histology and Mutation
Similarly to what we saw with the PTM data above, we see that cell lines again cluster according to their histology, sub-histology, and shared mutations. Again, we see that the cell line H2106 is the only NSCLC cell line that clusters with the SCLC cell lines. It is reassuring that we are seeing the same behavior for H2106 in these two independent datasets. Unlike in PTM space, all eight SCLC cell lines (including DMS53 and H446) share a branch here. We also see that cell lines cluster based on the RB1 mutation (six of the eight SCLC cell lines carry one), but less so on KRAS and EGFR mutations. Cell lines may cluster based on EGFR mutation in PTM space and not gene-expression space since EGFR is a kinase that affects phosphorylation levels in PTM space.

We also see two high-level clusters of genes that have high/low expression in SCLC cell lines and vice versa in NSCLC cell lines:

In [15]:
row_cluster_summary(mat_rna)

Data-Type,rna,mean in NSCLC,mean in SCLC
cluster,,,
1,658,0.16,-0.57
2,342,-0.20,0.71


### Enrichment Analysis of Gene Expression Clusters
Clicking a cluster in the row dendrogram of the expression Clustergram sends its genes to the Enrich widget, which runs enrichment analysis with [Enrichr](https://maayanlab.cloud/Enrichr/). Clicking an enriched term highlights its genes back in the Clustergram. We only use Enrich with gene expression: PTM rows are individual modification sites rather than genes, so a cluster of PTMs is not the kind of gene set that Enrichr expects. Enrich starts on the Gene Ontology Biological Process library, and its library menu also includes others used in the original analysis, such as KEGG, Disease Perturbations from GEO, and the MGI Mammalian Phenotype library.

At a high level, the original analysis found that the genes and proteins with higher PTM and expression levels in SCLC cell lines were associated with RNA processing and neuronal functions (e.g. neuron projection and axon guidance), in agreement with the neuronal characteristics of SCLC. Those higher in NSCLC cell lines were associated with cell motility, migration, and adhesion, in agreement with NSCLC cell lines forming adherent monolayers while SCLC cell lines grow in aggregates [(Doyle et al. 1990)](https://pmc.ncbi.nlm.nih.gov/articles/PMC329817/). Selecting the SCLC-high and NSCLC-high gene clusters above shows how much of this is visible in gene expression alone. See the original [Merged_SCLC_Cluster.ipynb](https://nbviewer.org/github/MaayanLab/CST_Lung_Cancer_Viz/blob/master/notebooks/Merged_SCLC_Cluster.ipynb) and [Merged_NSCLC_Cluster.ipynb](https://nbviewer.org/github/MaayanLab/CST_Lung_Cancer_Viz/blob/master/notebooks/Merged_NSCLC_Cluster.ipynb) notebooks for the full analysis.

## Merge PTM and Gene Expression Data
Above, we saw broadly similar patterns in lung cancer cell line behavior in PTM- and gene-expression-space:
* lung cancer cell lines cluster according to their histology, sub-histology, and shared mutations
* PTMs and genes form two large clusters with opposite expression in SCLC and NSCLC cell lines

Since we have processed the datasets similarly (e.g. Z-scored across cell lines) we can look for cross-data-type clusters by simply stacking our datasets vertically. Clustering this mixed data will help us identify potentially important correlations between expression, phosphorylation, methylation, and acetylation. Stacking all five modalities gives the same matrix as the merged `CST_CCLE_merge.txt` file in the Figshare dataset.

In [16]:
mat_merged = clustergram_matrix(stack(*PTM_TYPES, "rna"), "ptm_rna")
cgm_merged = dega.viz.Clustergram(matrix=mat_merged, width=700, height=750)

In [17]:
cgm_merged

Clustergram(category_colors={'phospho': '#FF0000', 'Rme1': '#800080', 'AcK': '#0000FF', 'Kme1': '#808080', 'rn…

### Cell Lines Cluster According to Histology and Mutation
As expected, we see that our cell lines again cluster according to their histology and mutations. We can now look at the two clusters of differentially regulated PTMs/genes across NSCLC and SCLC cell lines. Below are the breakdowns of data-types in the two large clusters at the top of the row dendrogram:

In [18]:
row_cluster_summary(mat_merged)

Data-Type,AcK,Kme1,Rme1,phospho,rna,mean in NSCLC,mean in SCLC
cluster,,,,,,,
1,185,23,269,356,321,-0.14,0.52
2,41,28,99,729,679,0.13,-0.47


We see that the cluster that is up-regulated in NSCLC cell lines is composed almost entirely of gene expression and phosphorylation data (89% of its rows). The cluster that is up-regulated in SCLC cell lines is composed mostly of phosphorylation, expression, arginine methylation, and lysine acetylation data.

### NKX2-1 Cluster
We find that the lung associated transcription factor NKX2-1 (which has been used as a biomarker in lung cancer, [Yang et al. 2012](https://pmc.ncbi.nlm.nih.gov/articles/PMC3494024/)) has similar gene expression and arginine methylation levels across lung cancer cell lines. Its expression row and its R121 methylation site are two rows apart in the merged Clustergram (with SFTA3 between them), close to SOX2. Search for NKX2-1 in the Clustergram to find them:

In [19]:
mat_merged.data.loc[["NKX2-1", "NKX2-1_Rme1_R121"]].T.corr().round(2)

,NKX2-1,NKX2-1_Rme1_R121
NKX2-1,1.00,0.77
NKX2-1_Rme1_R121,0.77,1.00


In the original analysis, NKX2-1 was clustered with a set of cytoskeletal and extracellular matrix related proteins (e.g. actin, myosin). This might point to a functionally important role for NKX2-1 regulation of the cytoskeleton and ECM in lung cancer, which has been previously proposed [(Yamaguchi et al. 2013)](https://doi.org/10.1016/j.ccr.2013.04.002).

## How the Sub-Histologies Cluster
To compare how the sub-histologies cluster in the PTM, gene expression, and merged Clustergrams, we cut each column dendrogram into its top three branches:

In [20]:
views = {"PTM": mat_ptm, "rna": mat_rna, "PTM + rna": mat_merged}
pd.concat(
    {view: column_branches(mat, n_clusters=3, by="Sub-Histology") for view, mat in views.items()},
    names=["view", "branch"],
)

Sub-Histology     NSCLC  SCLC  adenocarcinoma  \
view      branch                                
PTM       1           1     6               0   
          2           4     2               2   
          3           3     0              11   
rna       1           1     8               0   
          2           1     0               7   
          3           6     0               6   
PTM + rna 1           1     8               0   
          2           4     0               3   
          3           3     0              10   

Sub-Histology     bronchioloalveolar_adenocarcinoma  large_cell_carcinoma  \
view      branch                                                            
PTM       1                                       0                     0   
          2                                       0                     2   
          3                                       4                     0   
rna       1                                       0                     0   
          2                                       4                     0   
          3                                       0                     2   
PTM + rna 1                                       0                     0   
          2                                       0                     2   
          3                                       4                     0   

Sub-Histology     squamous_cell_carcinoma  
view      branch                           
PTM       1                             0  
          2                             0  
          3                             2  
rna       1                             0  
          2                             0  
          3                             2  
PTM + rna 1                             0  
          2                             0  
          3                             2

* **SCLC** cell lines share a branch with the NSCLC cell line H2106 in every view. In PTM space, the two SCLC cell lines without an RB1 mutation (DMS53 and H446) join an NSCLC branch instead.
* **Bronchioloalveolar adenocarcinoma** cell lines, a subtype of adenocarcinoma, always share a branch with most of the **adenocarcinoma** cell lines.
* In the PTM and merged views, the two **large cell carcinoma** cell lines group with cell lines annotated only as NSCLC, apart from most adenocarcinomas.
* The two **squamous cell carcinoma** cell lines (HCC15 and Lou-NH91) stay close to each other and are neighbors in the expression Clustergram. In the PTM and merged views, they join the adenocarcinoma-rich branch.

Overall, histology (SCLC versus NSCLC) is the strongest signal in every view, while the NSCLC sub-histologies are only partly separated, mainly adenocarcinomas versus large cell and unspecified NSCLC cell lines.

## Next Steps
The comparison of cell line clusters across omics depends on the number of clusters. A complementary view is a co-incidence plot of nearest neighbors, which compares each cell line's nearest neighbors across omics without cutting the dendrograms.

## References
* Original notebook: [CST_Data_Viz.ipynb](https://nbviewer.org/github/MaayanLab/CST_Lung_Cancer_Viz/blob/master/notebooks/CST_Data_Viz.ipynb) in [MaayanLab/CST_Lung_Cancer_Viz](https://github.com/MaayanLab/CST_Lung_Cancer_Viz)
* Data: Hornbeck P, Rikova K, Fernandez N, Ma'ayan A (2017). Clustergrammer Lung Cancer PTM Case Study Data. figshare. [https://doi.org/10.6084/m9.figshare.5339689.v2](https://doi.org/10.6084/m9.figshare.5339689.v2)
* Fernandez NF, et al. (2017). Clustergrammer, a web-based heatmap visualization and analysis tool for high-dimensional biological data. Scientific Data 4, 170151. [https://doi.org/10.1038/sdata.2017.151](https://doi.org/10.1038/sdata.2017.151)